# 05 — Performance and battery trade

Plot electrical power above stall, cruise/climb matching constraints, and 5S1P
versus 5S2P re-flown at their own masses. Larger packs require a new physical fit
and balance check and may violate hand-launch limits on the unchanged wing.


## Power and endurance accounting

**Inputs:** selected wing/polar, pack and propulsion efficiency assumptions,
requirements and payload rail loads. **Implementation:** `uavdesign/aero/polar.py`,
`uavdesign/endurance.py`, `uavdesign/sizing.py`, `design.battery_trade`.

With $q=\rho V^2/2$, the drag model is
$D=qS(C_{D0}+kC_L^2)$ and aerodynamic power is $DV$ [W]. Electrical input is
$P_e=DV/(\eta_p\eta_m\eta_{ESC})+P_{hotel}$, where $P_{hotel}$ includes the
modeled avionics and payload loads. The endurance function applies usable battery
energy and the configured Peukert correction; nominal $E/P_e$ alone is not its
complete result. The vertical stall marker bounds the usefulness of the smooth
attached-flow power curve.

Plain-text equations: `drag = 0.5 * density * speed^2 * area * (CD0 + k*CL^2)`;
`electrical_power = drag * speed / (prop_eff * motor_eff * ESC_eff) + hotel_power`.

Each 5S pack candidate changes aircraft mass and is evaluated on the fixed
baseline wing. More electrical energy also raises stall speed and structural load;
there is no CAD or handling approval implicit in the battery table. Coverage again
uses the estimated full-duration upper bound from notebook 01.

**External review:** establish propulsive efficiency versus airspeed, minimum-pack
thrust/current, cell temperature and voltage sag. Allocate climb, transit and
return energy before using any duration as survey time. The nominal sizing target
and actual discrete-pack prediction should both remain visible.


In [ ]:
import os
for key in ("OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[key] = "1"
os.environ.setdefault("MPLCONFIGDIR", "/tmp/uavdesign-notebook-matplotlib")
"""Locate the repo root so `import uavdesign` works without `pip install -e .`."""
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "uavdesign").is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

PLATFORM_DIR = REPO_ROOT / "platforms" / "survey_wing"
FIG_DIR = REPO_ROOT / "outputs" / "figures" / "survey_wing"
FIG_DIR.mkdir(parents=True, exist_ok=True)

import yaml
import numpy as np
import matplotlib.pyplot as plt

import json
import pandas as pd
from IPython.display import display, Markdown, Image
from uavdesign import design as D
from uavdesign import survey as sv
from uavdesign.atmosphere import G
studies = yaml.safe_load((PLATFORM_DIR / "notebook_studies.yaml").read_text())
d = D.run(solve_stability=False)
s = d.design["survey"]
cam = sv.Camera.from_dict(s["camera"])
from dataclasses import replace
acquisition_cam = replace(cam, max_fps=min(cam.max_fps, s["acquisition_rate_cap_hz"]))
pl = d.design["planform"]
req = d.design["requirements"]

def save(fig, name):
    fig.tight_layout()
    path = FIG_DIR / name
    fig.savefig(path, dpi=140)
    display(Image(filename=str(path)))
    plt.close(fig)


In [ ]:
from uavdesign.aero.polar import FixedWingAero
from uavdesign.aero import constraint as con
ac = FixedWingAero(d.selected.auw_kg, d.selected.wing_area_m2, d.selected.polar, d.rho)
inp = D.inputs_for(d)
speed = np.linspace(ac.stall_speed()*studies["power_speed_factors"][0],
                    d.selected.cruise_speed_m_s*studies["power_speed_factors"][1], studies["plot_points"])
efficiency = inp.prop_efficiency * inp.motor_efficiency * inp.esc_efficiency
power = ac.aero_power_w(speed)/efficiency + inp.avionics_power_w + inp.payload_power_w
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(speed, power)
axes[0].axvspan(speed.min(), ac.stall_speed(), alpha=.2, color="red", label="Below stall")
axes[0].axvline(d.selected.cruise_speed_m_s, ls="--", label="Design cruise")
axes[0].set(xlabel="Airspeed [m/s]", ylabel="Electrical power [W]"); axes[0].legend()
ws = np.linspace(ac.wing_loading_n_m2*studies["wing_loading_factors"][0],
                 ac.wing_loading_n_m2*studies["wing_loading_factors"][1], studies["plot_points"])
v = d.selected.cruise_speed_m_s
for label, tw in (("Cruise", con.cruise_tw(ws,d.rho,v,ac.polar.cd0,ac.polar.k)),
                  ("Climb", con.climb_tw(ws,d.rho,v,ac.polar.cd0,ac.polar.k,req["climb_rate_m_s"]))):
    axes[1].plot(ws, con.power_per_mass_w_kg(tw,v,inp.prop_efficiency), label=label)
axes[1].axvline(con.launch_ws_max(d.rho, req["hand_launch_speed_m_s"],ac.polar.cl_max,req["launch_margin"]),ls="--",label="Hand-launch W/S maximum")
axes[1].set(xlabel="Wing loading [N/m²]",ylabel="Required shaft power [W/kg]"); axes[1].legend()
save(fig, "05_power_matching.png")
rows = D.battery_trade(d, parallels=studies["battery_parallel_counts"])
for row in rows:
    row["coverage_upper_bound_ha"] = sv.coverage_ha(cam,s["agl_m"],row["cruise_speed_m_s"],s["side_overlap"],row["endurance_min"]*60,s["turn_efficiency"])
display(pd.DataFrame(rows))
fig, axes = plt.subplots(1,2,figsize=(9,4))
for ax, key, label in ((axes[0],"endurance_min","Endurance [min]"),(axes[1],"coverage_upper_bound_ha","Survey coverage upper bound [ha]")):
    ax.bar([r["pack"] for r in rows],[r[key] for r in rows]); ax.set_ylabel(label)
save(fig,"05_pack_trade.png")
